# Tome 3, Chapter 3: Generating Functions (Python companion)

Exercises first, solutions at the end. We let `sympy` manipulate generating functions, simulate Galton–Watson processes and compare their extinction rates with the smallest fixed point of the PGF, look at the heavy tail of a critical family tree, and meet random binary search trees, the data structure behind the quicksort of Chapter 2 and another home of the Catalan numbers.

*Before each solution, predict what the output will look like.*

In [ ]:
import math
from fractions import Fraction
import random
from collections import Counter
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
rng = np.random.default_rng(3)
random.seed(3)

## Exercises

**P3.1 (★) Generating functions with sympy.** Compute symbolically the PGFs of the classical laws, check Corollary 3.5 (sums of Poissons) and the thinning of Exercise 3.1. Then check by simulation that, in the thinning, the numbers of hatched and unhatched eggs are *independent*: compare $P(H=i,U=j)$ with $P(H=i)P(U=j)$.

In [ ]:
# Your code here


**P3.2 (★★) Extinction, simulated.** Simulate $20\,000$ Galton–Watson processes for each of the offspring laws $(\frac14,\frac14,\frac12)$, $0$ or $3$ children with probability $\frac12$ (Exercise 3.2), Lotka's law (Exercise 3.3) and Poisson$(2)$ (Exercise 3.8), declaring extinction if $Z_n=0$ within $60$ generations and survival if $Z_n$ exceeds $1000$. Compare the extinction frequency with the smallest fixed point of $G$ (Theorem 3.8), computed by iterating $q_{n+1}=G(q_n)$.

In [ ]:
# Your code here


**P3.3 (★★) A critical family tree.** For the offspring law $p_k=2^{-(k+1)}$, simulate the total progeny $T$ $10^5$ times, compare its law with $C_{n-1}2^{-(2n-1)}$ (Theorem 3.11), and plot $P(T\ge n)$ on log-log axes. What is the slope, and what does it say about $\mathbb E[T]$?

In [ ]:
# Your code here


**P3.4 (★) Catalan numbers three ways.** Get the first $15$ Catalan numbers from the series of $\frac{1-\sqrt{1-4x}}{2x}$, from the recursion of Proposition 3.10, and by enumerating Dyck paths for small $n$. Check $C_n\sim\frac{4^n}{\sqrt\pi n^{3/2}}$.

In [ ]:
# Your code here


**P3.5 (★) First passage.** For a walk with $p=0.4$, simulate the first passage time to $1$ (capped at $10^4$ steps) and compare $P(\tau<\infty)$ and the law of $\tau$ with Exercise 3.6. Extract the law of $\tau$ from the series of $F(s)$ with sympy.

In [ ]:
# Your code here


**P3.6 (★★) Random binary search trees.** Insert a uniformly random permutation of $\{1,\dots,n\}$ into a binary search tree. (a) Check that the number of comparisons made by all the insertions has the same law as the comparisons of randomised quicksort (Exercise 2.3), and that the average depth of a node is $2(1+\frac1n)H_n-4\approx2\ln n-2.85$. (b) Check that the number of possible *shapes* of a BST with $n$ nodes is $C_n$. (c) Estimate the mean height for $n=10^3,\dots,10^5$ and compare with Devroye's $4.311\ln n$.

In [ ]:
# Your code here


**P3.7 (★) Poisson branching near criticality.** Compute the survival probability $\rho(c)$ of a Poisson$(c)$ Galton–Watson process for $c\in[0,3]$ and plot it with the approximation $2(c-1)$ near $c=1$ (Exercise 3.8).

In [ ]:
# Your code here


---
## Solutions

### P3.1

In [ ]:
s, p, lam, mu, n = sp.symbols('s p lambda mu n', positive=True)
k = sp.symbols('k', integer=True, nonnegative=True)
G_poisson = sp.simplify(sp.summation(sp.exp(-lam) * lam**k / sp.factorial(k) * s**k, (k, 0, sp.oo)))
G_geom = p * s / (1 - (1 - p) * s)   # closed form; check its Taylor coefficients below
assert all(sp.simplify(sp.series(G_geom, s, 0, 8).removeO().coeff(s, j) - p * (1 - p)**(j - 1)) == 0 for j in range(1, 8))
print('Poisson PGF:', G_poisson, '   geometric PGF:', G_geom)
print('Poisson(lam) + Poisson(mu):', sp.simplify(G_poisson * G_poisson.subs(lam, mu)))
print('thinned Poisson:', sp.simplify(G_poisson.subs(s, 1 - p + p * s)))
print('E[X], Var X for the geometric law:', sp.simplify(sp.diff(G_geom, s).subs(s, 1)),
      sp.simplify((sp.diff(G_geom, s, 2) + sp.diff(G_geom, s) - sp.diff(G_geom, s)**2).subs(s, 1)))

m = 400_000
N = rng.poisson(6.0, m)
H = rng.binomial(N, 0.3)
U = N - H
joint = Counter(zip(H.tolist(), U.tolist()))
pH, pU = Counter(H.tolist()), Counter(U.tolist())
worst = max(abs(joint[(i, j)] / m - pH[i] / m * pU[j] / m) for i in range(6) for j in range(10))
print(f'max |P(H=i,U=j) - P(H=i)P(U=j)| over i<6, j<10: {worst:.4f}')
print('means', H.mean(), U.mean(), '(theory 1.8 and 4.2); correlation', np.corrcoef(H, U)[0, 1])

Sympy recovers $e^{\lambda(s-1)}$, $\frac{ps}{1-(1-p)s}$, the sum rule and the thinning rule, and the mean and variance of the geometric law from $G'(1)$ and $G''(1)+G'(1)-G'(1)^2$. The simulation shows something the PGF of $H$ alone does not: hatched and unhatched counts are *independent* (correlation $\approx0$, joint probabilities equal to products). For a binomial number of eggs they would be negatively correlated; the Poisson law is the only one with this property, a fact at the heart of Poisson processes (Chapter 9).

### P3.2

In [ ]:
def simulate_gw(sampler, runs=20_000, gens=60, cap=1000):
    extinct = 0
    for _ in range(runs):
        z = 1
        for _ in range(gens):
            z = int(sampler(z).sum()) if z > 0 else 0
            if z == 0: extinct += 1; break
            if z > cap: break
    return extinct / runs
def smallest_fixed_point(G):
    q = 0.0
    for _ in range(100_000):
        q = G(q)
    return q
a, r, p0 = 0.2126, 0.5893, 0.4825
def lotka_sampler(z):
    die = rng.random(z) < p0 / (p0 + a / (1 - r))            # renormalised: sum of published p_k is 1.00015
    return np.where(die, 0, rng.geometric(1 - r, z))
laws = {
    '(1/4,1/4,1/2)': (lambda z: rng.choice(3, z, p=[.25, .25, .5]), lambda s: .25 + .25 * s + .5 * s * s),
    '0 or 3': (lambda z: 3 * (rng.random(z) < .5), lambda s: .5 + .5 * s**3),
    'Lotka': (lotka_sampler, lambda s: (p0 + a * s / (1 - r * s)) / (p0 + a / (1 - r))),
    'Poisson(2)': (lambda z: rng.poisson(2.0, z), lambda s: math.exp(2 * (s - 1))),
}
for name, (sampler, G) in laws.items():
    q = smallest_fixed_point(G)
    f = simulate_gw(sampler)
    print(f'{name:14s} extinction: simulated {f:.4f} +- {1.96*math.sqrt(f*(1-f)/20000):.4f}, fixed point {q:.4f}')
print('1/golden ratio =', (math.sqrt(5) - 1) / 2)

Each simulated frequency lands within its error bar of the smallest fixed point: $\frac12$, $\frac1\varphi\approx0.618$, $0.82$ for Lotka (after renormalising the published law, which sums to $1.00015$), and $1-0.797=0.203$ for Poisson$(2)$. The two stopping rules are harmless: a population that has reached $1000$ individuals dies out with probability at most $q^{1000}$, since the $1000$ subtrees are independent, and a population alive after $60$ generations is almost surely large.

### P3.3

In [ ]:
def total_progeny(cap=10**6):
    alive, total = 1, 1
    while alive and total < cap:
        children = int(rng.geometric(0.5, alive).sum() - alive)   # numpy geometric starts at 1: subtract 1 each
        alive = children; total += children
    return total
m = 100_000
T = np.array([total_progeny() for _ in range(m)])
catalan = lambda n: math.comb(2 * n, n) // (n + 1)
print(' n   simulated   C_(n-1) 2^-(2n-1)')
for nn in range(1, 8):
    print(f'{nn:2d}  {np.mean(T == nn):.4f}     {catalan(nn - 1) / 2**(2*nn - 1):.4f}')
ns = np.unique(np.logspace(0, 4, 30).astype(int))
tail = [np.mean(T >= nn) for nn in ns]
slope = np.polyfit(np.log(ns[5:]), np.log(tail[5:]), 1)[0]
fig, ax = plt.subplots(figsize=(6, 3))
ax.loglog(ns, tail, 'o', ms=3, label=f'$P(T\\geq n)$, slope {slope:.2f}')
ax.loglog(ns, 1 / np.sqrt(np.pi * ns), '--', label='$1/\\sqrt{\\pi n}$')
ax.set_xlabel('n'); ax.legend(); plt.show()
print('sample mean of T (unstable!):', T.mean(), '  max:', T.max())

The law matches $C_{n-1}2^{-(2n-1)}$: half the trees are a single individual, an eighth have two, and so on. The tail $P(T\ge n)$ decreases like $n^{-1/2}$ (summing $\frac1{2\sqrt\pi}k^{-3/2}$ for $k\ge n$ gives $\frac1{\sqrt{\pi n}}$), so $\mathbb E[T]=\sum_nP(T\ge n)=\infty$. The sample mean is therefore meaningless: it is dominated by the few largest trees and changes wildly from run to run. Every tree is finite, but there is no typical size.

### P3.4

In [ ]:
x = sp.symbols('x')
series = sp.series((1 - sp.sqrt(1 - 4 * x)) / (2 * x), x, 0, 15).removeO()
from_series = [series.coeff(x, n) for n in range(15)]
rec = [1]
for n in range(14):
    rec.append(sum(rec[k] * rec[n - k] for k in range(n + 1)))
def dyck(n):
    count = 0
    for steps in range(2 ** (2 * n)):
        h = 0
        for i in range(2 * n):
            h += 1 if steps >> i & 1 else -1
            if h < 0: break
        else:
            count += h == 0
    return count
print('series   :', from_series)
print('recursion:', rec)
print('Dyck paths (n <= 9):', [dyck(n) for n in range(10)])
for n in [10, 100, 1000]:
    print(f'n = {n}: C_n / (4^n / (sqrt(pi) n^1.5)) = {float(Fraction(catalan(n), 4**n)) * math.sqrt(math.pi) * n**1.5:.5f}')

Three computations, one sequence $1,1,2,5,14,42,\dots$. The ratio with $\frac{4^n}{\sqrt\pi n^{3/2}}$ tends to $1$ slowly, like $1-\frac9{8n}$; the next terms of the expansion come from Stirling's series.

### P3.5

In [ ]:
p = 0.4
q = 1 - p
def first_passage(cap=10_000):
    pos = 0
    for t in range(1, cap + 1):
        pos += 1 if random.random() < p else -1
        if pos == 1: return t
    return None
m = 50_000
taus = [first_passage() for _ in range(m)]
hit = [t for t in taus if t is not None]
print(f'P(tau < inf): simulated {len(hit)/m:.4f}, theory p/q = {p/q:.4f}')
s = sp.symbols('s')
F = (1 - sp.sqrt(1 - 4 * sp.Rational(2, 5) * sp.Rational(3, 5) * s**2)) / (2 * sp.Rational(3, 5) * s)
coeffs = sp.series(F, s, 0, 12).removeO()
cnt = Counter(hit)
for t in [1, 3, 5, 7, 9, 11]:
    print(f'P(tau = {t:2d}): theory {float(coeffs.coeff(s, t)):.5f}   simulated {cnt[t]/m:.5f}')

With a downward drift the walk reaches $1$ only with probability $\frac pq=\frac23$, and only at odd times; the coefficients of $F(s)$ are $P(\tau=2n+1)=C_n\,p^{n+1}q^n$, Catalan again, because a first passage to $1$ is a Dyck path followed by one up-step.

### P3.6

In [ ]:
def bst_stats(perm):
    # insert keys into a BST stored as dicts; return total comparisons, depths, height
    left, right = {}, {}
    root = perm[0]
    depth = {root: 0}
    comparisons = 0
    for key in perm[1:]:
        node, d = root, 0
        while True:
            comparisons += 1; d += 1
            nxt = left if key < node else right
            if node in nxt: node = nxt[node]
            else:
                nxt[node] = key; depth[key] = d; break
    return comparisons, depth, max(depth.values())
n = 1000
H = sum(1 / k for k in range(1, n + 1))
runs = [bst_stats(list(rng.permutation(n)))[0] for _ in range(2000)]
print(f'BST insertion comparisons: mean {np.mean(runs):.0f}, quicksort formula 2(n+1)H_n - 4n = {2*(n+1)*H - 4*n:.0f}')
print(f'average depth {np.mean(runs)/n:.2f}, 2(1+1/n)H_n - 4 = {2*(1+1/n)*sum(1/j for j in range(1, n+1)) - 4:.2f}, 2 ln n = {2*math.log(n):.2f}')

def shapes(n):
    if n == 0: return {()}
    out = set()
    for perm in __import__('itertools').permutations(range(n)):
        left, right = {}, {}
        root = perm[0]
        for key in perm[1:]:
            node = root
            while True:
                nxt = left if key < node else right
                if node in nxt: node = nxt[node]
                else: nxt[node] = key; break
        def shape(v):
            return (shape(left[v]) if v in left else None, shape(right[v]) if v in right else None)
        out.add(shape(root))
    return out
print('shapes:', [len(shapes(k)) for k in range(1, 8)], ' Catalan:', [catalan(k) for k in range(1, 8)])

import sys
sys.setrecursionlimit(10000)
for n in [10**3, 10**4, 10**5]:
    hs = [bst_stats(list(rng.permutation(n)))[2] for _ in range(20 if n < 10**5 else 5)]
    print(f'n = {n:>6}: mean height {np.mean(hs):.1f},  4.311 ln n = {4.311*math.log(n):.1f},  2 ln n = {2*math.log(n):.1f}')

Inserting a random permutation into a BST makes exactly the comparisons of quicksort with the same first pivots (the root is the first pivot, and each key is compared with the pivots of the subarrays it falls in), so the totals agree with $2(n+1)H_n-4n$, and dividing by $n$ (the depth of a node is the number of comparisons made to insert it) gives the average depth $2(1+\frac1n)H_n-4$, which is $2\ln n$ up to a constant $2\gamma-4\approx-2.85$. The shapes of BSTs with $n$ nodes are the binary trees with $n$ nodes, counted by $C_n$ (a binary tree is empty or a root with two binary trees: $C=1+xC^2$). The *height* is larger than the typical depth: Devroye (1986) proved that it is $\sim4.311\ln n$, where $4.311\ldots$ solves $c\ln\frac{2e}c=1$; the convergence is slow, and at $n=10^5$ the mean height is still visibly below $4.311\ln n$ because of a negative $\ln\ln n$ correction (Reed, 2003).

### P3.7

In [ ]:
def rho(c):
    if c <= 1: return 0.0
    r = 1.0
    for _ in range(100_000):
        r = 1 - math.exp(-c * r)
    return r
cs = np.linspace(0, 3, 301)
rs = [rho(c) for c in cs]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(cs, rs, label='survival probability $\\rho(c)$')
ax.plot(cs[100:140], 2 * (cs[100:140] - 1), '--', label='$2(c-1)$')
ax.set_xlabel('mean offspring c'); ax.legend(); plt.show()
for c in [1.01, 1.1, 2.0, 3.0]:
    print(f'c = {c}: rho = {rho(c):.5f}, 2(c-1)/c^2 = {2*(c-1)/c**2:.5f}')

Survival is impossible up to $c=1$ and then rises linearly, with slope $2$: a *phase transition*, continuous but with a corner. Chapter 12 shows that the same curve gives the fraction of vertices in the giant component of a random graph with average degree $c$: exploring a sparse random graph from a vertex looks like a Poisson$(c)$ branching process, until the graph starts to run out of new vertices.